In [1]:
import sys
import os

# Add the root thesis folder path to sys.path 
project_root = '/srv/homes/onbo10/thesis_Ons'
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from TopDown_kpts_detection import UnifiedSurgicalPipeline
from HRNet_YOLO.HRNet_one_instance.training_HRNET_YOLO import *
from evaluation_utils import run_test_on_yolo_format
from mmpose.apis import init_model
from ultralytics import YOLO
import torch

* Load the object detection model:

In [6]:
det_model = YOLO("/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/Yolo/runs_yolo/surgpose_exp1/weights/best.pt")
det_model.eval()

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

* Set the test dataset paths and the device

In [7]:
test_img_dir= '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/images/test'
test_lbl_dir ='/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/yolo_formated_surgpose_kpts/labels/test'
device = get_device()

* Evaluate the pipline with Vitpose

In [5]:

config_file = '/srv/homes/onbo10/thesis_Ons/mmpose/configs/body_2d_keypoint/topdown_heatmap/surgpose/vitpose_surg_7kpt.py'
checkpoint_file = '/srv/homes/onbo10/thesis_Ons/ViTPose/work_dirs/vitpose_base_surg_experiment_1/best_coco_AP_epoch_180.pth'
model_type_vitpose='vitpose'
pose_model_vitpose = init_model(config_file, checkpoint_file, device=device)
pose_model_vitpose.eval()

Loads checkpoint by local backend from path: /srv/homes/onbo10/thesis_Ons/ViTPose/work_dirs/vitpose_base_surg_experiment_1/best_coco_AP_epoch_180.pth


TopdownPoseEstimator(
  (data_preprocessor): PoseDataPreprocessor()
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=(2, 2))
    )
    (drop_after_pos): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (ln1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): MultiheadAttention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
          (out_drop): DropPath()
          (gamma1): Identity()
        )
        (ln2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (ffn): FFN(
          (layers): Sequential(
            (0): Sequential(
              (0): Linear(in_features=768, out_features=3072, bias=True)
              (1): GELU(approximate='none')
         

In [6]:

vitpose_pipeline = UnifiedSurgicalPipeline(det_model,pose_model_vitpose,model_type_vitpose, device= device)


In [7]:
ap_vitpose= run_test_on_yolo_format(vitpose_pipeline,test_img_dir,test_lbl_dir,device)

  0%|          | 0/2004 [00:00<?, ?it/s]/srv/homes/onbo10/.local/lib/python3.8/site-packages/ultralytics/utils/metrics.py:183: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = torch.tensor(sigma, device=kpt1.device, dtype=kpt1.dtype)  # (17, )
100%|██████████| 2004/2004 [01:58<00:00, 16.92it/s]


--- VITPOSE FINAL EVAL ---
Precision:    0.9925
Recall:       0.9923
mAP@50:    0.9908
mAP@50-95: 0.9850


* Evaluate the pipeline with HRNet

In [8]:
cfg_file = '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/HRNet_one_instance/w32_256x192_adam_lr1e-3_out7-finetune.yaml'
model_weights= '/srv/homes/onbo10/thesis_Ons/HRNet_YOLO/HRNet_one_instance/Experiment1/training_chekpoints2026-01-11_18-05-32/model_epoch200.pth'

pose_model_hrnet= load_pretrained_HRNet(cfg_file,model_weights, finetuned=True)
model_type_hrnet='hrnet'
pose_model_hrnet.to(device)
pose_model_hrnet.eval()

PoseHighResolutionNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(i

In [9]:
hrnet_pipeline = UnifiedSurgicalPipeline(det_model,pose_model_hrnet,model_type_hrnet, device= device)


In [10]:
ap_hrnet= run_test_on_yolo_format(hrnet_pipeline,test_img_dir,test_lbl_dir,device)

  0%|          | 0/2004 [00:00<?, ?it/s]/srv/homes/onbo10/.local/lib/python3.8/site-packages/ultralytics/utils/metrics.py:183: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma = torch.tensor(sigma, device=kpt1.device, dtype=kpt1.dtype)  # (17, )
100%|██████████| 2004/2004 [03:00<00:00, 11.11it/s]


--- HRNET FINAL EVAL ---
Precision:    0.9915
Recall:       0.9913
mAP@50:    0.9912
mAP@50-95: 0.9227
